<div style="border-left:6px solid #b30338; padding:8px 16px;">

# MCDI505 — Análisis de Series de Tiempo
## Caso Transversal del Curso
### Documento base · Serie temporal, contexto y decisiones a fundamentar

**Magíster en Ciencia de Datos · UNAB Online**  
**Alcance:** Semanas 1 a 4 · un único caso, cuatro entregas encadenadas

</div>

---

Este documento **no es una sesión de contenidos**: es el **contrato de trabajo** del proyecto que usted desarrollará durante las próximas cuatro semanas. Léalo completo antes de la primera entrega y vuelva a él cada semana.

### Qué encontrará aquí

| Sección | Contenido |
|---|---|
| §1 | Cómo funciona el caso transversal: entregas, encadenamiento y evaluación |
| §2–§3 | La organización, el problema de negocio y la pregunta que debe responder |
| §4–§5 | La serie provista: diccionario de variables, periodo, frecuencia y procedencia |
| §6 | Auditoría de integridad de los datos (obligatoria antes de modelar) |
| §7 | Primera caracterización visual |
| §8 | Protocolo de partición cronológica y sellado del conjunto de prueba |
| §9 | Modelos de referencia (*baselines*) y métricas: la vara que deberá superar |
| §10 | **Las decisiones que deberá fundamentar, semana por semana** |
| §11–§12 | Reglas de entrega, reproducibilidad y rúbrica |
| §13 | Entregable de la Semana 1 |
| §14 | Uso excepcional de un conjunto de datos propio |

### Requisitos técnicos
Python 3.10 o superior con `numpy`, `pandas`, `matplotlib` y `statsmodels`. El único archivo externo necesario es `consumo_electrico_litoral.csv`, disponible en el aula de Canvas.

---
# 1. Cómo funciona el caso transversal

Las cuatro entregas **no son ejercicios independientes**: cada una toma como insumo las decisiones documentadas en la anterior. Una decisión mal fundamentada en la Semana 2 arrastra consecuencias hasta la Semana 4, exactamente como ocurre en un proyecto real.

```
 SEMANA 1                SEMANA 2                SEMANA 3               SEMANA 4
 ─────────               ─────────               ─────────              ─────────
 Caracterización    →    Estacionariedad    →    Modelamiento     →    Evaluación y
 de la serie             y transformaciones      SARIMA                pronóstico
      │                        │                      │                     │
      ▼                        ▼                      ▼                     ▼
 ¿Qué tiene esta        ¿Qué transformación     ¿Qué órdenes          ¿Es mejor que un
 serie dentro?          la deja modelable?      (p,d,q)(P,D,Q,s)?     modelo trivial?
      │                        │                      │                     │
      └──── decide ────────────┴──── decide ──────────┴──── decide ─────────┘
            d y D                    p, q, P, Q            recomendación
                                                            operacional
```

### Criterio de calidad transversal

> Lo que se evalúa **no es acertar el pronóstico**, sino la **calidad de la fundamentación**. Un modelo con error moderado bien justificado vale más que uno con error bajo elegido a ciegas.

Cada decisión técnica debe presentarse en este formato:

| Elemento | Ejemplo |
|---|---|
| **Decisión** | Aplicar logaritmo antes de diferenciar. |
| **Evidencia** | La desviación estándar móvil crece de forma aproximadamente proporcional al nivel (gráfico 3.2). |
| **Alternativa descartada** | Box-Cox con λ estimado: entrega λ ≈ 0,08, cercano a 0, y el logaritmo es más interpretable. |
| **Consecuencia** | Los pronósticos se generan en escala logarítmica y requieren retransformación. |

### Calendario de entregas

| Semana | Entregable | Formato | Retroalimentación |
|---|---|---|---|
| 1 | Caracterización y auditoría de la serie | Notebook `.ipynb` + HTML | 5 días hábiles |
| 2 | Diagnóstico de estacionariedad y transformaciones | Notebook `.ipynb` + HTML | 5 días hábiles |
| 3 | Identificación, estimación y validación del modelo | Notebook `.ipynb` + HTML | 5 días hábiles |
| 4 | Pronóstico, evaluación e informe ejecutivo | Notebook + informe de 2 páginas | 7 días hábiles |

---
# 2. Configuración del entorno y carga de la serie

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import acf, adfuller, kpss

warnings.filterwarnings("ignore")

SEMILLA = 505
np.random.seed(SEMILLA)

plt.rcParams.update({
    "figure.figsize": (11, 3.6), "figure.dpi": 110,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 11, "font.size": 9, "lines.linewidth": 1.3,
})

ARCHIVO = "consumo_electrico_litoral.csv"
CANDIDATOS = [Path(ARCHIVO), Path("datos") / ARCHIVO, Path("..") / "datos" / ARCHIVO,
              Path("data") / ARCHIVO]


def cargar_caso():
    '''Carga la serie del caso transversal desde alguna de las rutas habituales.'''
    for ruta in CANDIDATOS:
        if ruta.exists():
            df = pd.read_csv(ruta, parse_dates=["fecha"])
            print(f"Datos cargados desde: {ruta.resolve()}")
            return df.set_index("fecha").asfreq("MS")
    raise FileNotFoundError(
        f"No se encontró '{ARCHIVO}'.\n"
        "Descárguelo desde el aula de Canvas y déjelo junto a este notebook "
        "o dentro de una carpeta 'datos/'."
    )


df = cargar_caso()
print(f"\nPython {sys.version.split()[0]} · pandas {pd.__version__} · statsmodels {sm.__version__}")
print(f"Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas")
df.head()

---
# 3. El caso: contexto organizacional

> ### 🏢 La organización
>
> **Distribuidora Eléctrica del Litoral (DEL)** opera la red de distribución de un conjunto de comunas costeras y del interior de la Región de Valparaíso. Su **segmento comercial** —oficinas, comercio minorista, hotelería, servicios— representa cerca de un tercio de la energía que la empresa distribuye y es el de mayor variabilidad estacional.
>
> ### 💼 La decisión que depende de usted
>
> Cada año, en diciembre, DEL debe comprometer ante el Coordinador Eléctrico Nacional los **volúmenes de energía que contratará para el año siguiente, mes a mes**. Estos contratos son vinculantes y su desviación tiene costo:
>
> - **Contratar de menos** obliga a comprar la diferencia en el mercado *spot*, a precios volátiles que históricamente superan entre 2 y 4 veces el precio contratado.
> - **Contratar de más** implica pagar energía que no se consume, sin posibilidad de reventa.
>
> El costo del error es **asimétrico**: subestimar es más caro que sobreestimar, pero ambos duelen.
>
> ### ❓ La pregunta del caso
>
> **¿Cuál es el consumo mensual esperado del segmento comercial para los próximos 12 meses, con qué grado de incertidumbre, y qué volumen debería contratar DEL en consecuencia?**
>
> La Gerencia de Planificación no le pide únicamente un número: le pide un pronóstico **con intervalos**, una **justificación auditable** de cómo lo obtuvo, y una **recomendación** que reconozca la asimetría del costo.

### Antecedentes relevantes del periodo observado

La serie cubre diez años y contiene tres hechos que usted deberá considerar explícitamente:

1. **Abril de 2019.** Entró en operación un nuevo centro comercial y su zona de servicios asociada, lo que incorporó alrededor de 900 clientes comerciales a la red en un solo trimestre.
2. **Marzo de 2020 a diciembre de 2021.** Las restricciones sanitarias por COVID-19 redujeron drásticamente la actividad comercial. El consumo cayó con fuerza y se recuperó de forma gradual a lo largo de casi dos años.
3. **Noviembre de 2017.** Una recalibración de medidores en un sector de la red produjo un registro anómalo puntual, reconocido por el área de operaciones.

Ninguno de estos hechos está marcado en los datos. **Detectarlos y decidir qué hacer con ellos es parte del trabajo.**

---
# 4. Procedencia de los datos y diccionario de variables

### 4.1 Procedencia y advertencia metodológica

> ⚠️ **Estos datos son sintéticos.** Fueron generados con un proceso estocástico calibrado para reproducir la estructura de una serie real de consumo eléctrico comercial: tendencia de crecimiento, estacionalidad multiplicativa, un componente de nivel no estacionario, un shock exógeno prolongado, valores faltantes y un dato atípico.
>
> Se utiliza un conjunto sintético por tres razones deliberadas:
>
> 1. **Trazabilidad de la evaluación.** El proceso generador es conocido por el equipo docente, de modo que la retroalimentación puede indicarle si su diagnóstico fue correcto y no solo si fue plausible.
> 2. **Comparabilidad.** Todos los estudiantes trabajan sobre la misma serie, lo que hace comparables las decisiones y permite discutirlas en conjunto.
> 3. **Confidencialidad.** Los datos de consumo por segmento son información comercial sensible y no son de acceso público.
>
> El proceso generador **se revelará en la retroalimentación de la Semana 4**. Hasta entonces, trátelo como una serie real: usted no conoce la verdad, solo la evidencia.

### 4.2 Diccionario de variables

| Variable | Tipo | Unidad | Descripción | Rol en el análisis |
|---|---|---|---|---|
| `fecha` | fecha (`datetime64`) | — | Primer día del mes de medición. Índice de la serie. | Índice temporal |
| `consumo_gwh` | numérica continua | GWh | Energía facturada al segmento comercial en el mes. | **Variable objetivo** |
| `clientes_comerciales` | numérica discreta | conteo | Empalmes comerciales activos al cierre del mes. | Covariable / exógena |
| `temperatura_media_c` | numérica continua | °C | Temperatura media mensual de la zona de concesión. | Covariable / exógena |
| `dias_habiles` | numérica discreta | días | Días hábiles del mes (excluye sábados, domingos y festivos). | Covariable / exógena |

### 4.3 Metadatos de la serie

| Atributo | Valor |
|---|---|
| Periodo de observación | Enero de 2015 – diciembre de 2024 |
| Frecuencia temporal | Mensual (`MS`, inicio de mes) |
| Longitud | 120 observaciones |
| Periodo estacional | $s = 12$ |
| Valores faltantes | 3 en `consumo_gwh` (fallas de telemetría) |
| Zona horaria / calendario | Chile continental |
| Última actualización | 15 de enero de 2025 |

> ### 🔍 Univariada o multivariada
> La serie **objetivo** es univariada: una sola variable indexada en el tiempo, `consumo_gwh`. El conjunto de datos, en cambio, es **multivariado**: incluye tres covariables medidas en paralelo.
>
> Las semanas 1 a 4 se resuelven con un enfoque **univariado** (ARIMA / SARIMA). Las covariables cumplen dos funciones: **explicar** lo que observa (por ejemplo, ¿la estacionalidad se debe a la temperatura?) y habilitar una **extensión opcional** con SARIMAX en la Semana 4. Usarlas para explicar es obligatorio; usarlas para modelar es voluntario.

In [ ]:
print("=" * 74)
print("FICHA TÉCNICA DE LA SERIE")
print("=" * 74)
print(f"Periodo        : {df.index.min():%Y-%m} a {df.index.max():%Y-%m}")
print(f"Frecuencia     : {df.index.freqstr}  (mensual, inicio de mes)")
print(f"Observaciones  : {len(df)}   →   {len(df)/12:.0f} años completos")
print(f"Variable objetivo: consumo_gwh (GWh)")
print("\nEstadísticos descriptivos:")
display(df.describe().T.round(2))
print("\nTipos de dato y completitud:")
display(pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "no nulos": df.notna().sum(),
    "nulos": df.isna().sum(),
    "% nulos": (df.isna().mean() * 100).round(1),
}))

---
# 5. Auditoría de integridad

Antes de graficar y mucho antes de modelar, hay que responder cuatro preguntas sobre la **estructura** del índice temporal. Un análisis técnicamente impecable sobre datos mal indexados no vale nada.

| Pregunta | Por qué importa |
|---|---|
| ¿El índice es estrictamente creciente y sin duplicados? | Un duplicado o un desorden invalida todo cálculo de rezagos. |
| ¿Los intervalos son equiespaciados? | ACF, diferenciación y modelos ARIMA suponen espaciado constante. |
| ¿Hay huecos en el calendario? | Un mes ausente no es lo mismo que un mes con valor nulo. |
| ¿Hay valores nulos o imposibles? | Un consumo negativo o cero delata un error de registro. |

La función siguiente es parte de su caja de herramientas: **úsela en cada entrega**.

In [ ]:
def auditar_serie(df, columna, freq_esperada="MS", freq_periodos=12, umbral_z=3.5):
    '''Auditoria de integridad estructural de una serie temporal.

    Verifica: monotonia e unicidad del indice, equiespaciado, huecos de
    calendario, nulos y valores no positivos. Devuelve un dict con hallazgos.
    '''
    idx = df.index
    hallazgos = {}

    hallazgos["indice_ordenado"] = bool(idx.is_monotonic_increasing)
    hallazgos["indice_unico"] = bool(idx.is_unique)

    # equiespaciado: el indice debe coincidir EXACTAMENTE con el calendario
    # regular de la frecuencia declarada. Comparar diferencias en dias no sirve:
    # los meses tienen 28, 29, 30 o 31 dias y aun asi son equiespaciados en la
    # escala mensual, que es la escala en que opera el modelo.
    completo = pd.date_range(idx.min(), idx.max(), freq=freq_esperada)
    hallazgos["equiespaciado"] = bool(idx.equals(completo))

    faltan_fechas = completo.difference(idx)
    hallazgos["fechas_ausentes"] = list(faltan_fechas.strftime("%Y-%m"))

    s = df[columna]
    hallazgos["n_nulos"] = int(s.isna().sum())
    hallazgos["fechas_nulas"] = list(s[s.isna()].index.strftime("%Y-%m"))
    hallazgos["n_no_positivos"] = int((s <= 0).sum())

    # --- deteccion de atipicos ---
    # Comparar cada dato con el mes anterior o con el mismo mes del ano previo
    # confunde el atipico con la estacionalidad. El procedimiento correcto aisla
    # el componente irregular en tres pasos:
    #   1. media movil centrada de 12 meses  -> estima el nivel (tendencia-ciclo)
    #   2. razon serie/nivel                 -> deja estacionalidad + irregular
    #   3. dividir por la mediana del mes    -> deja solo el irregular
    # Sobre el irregular se aplica un z robusto (mediana y MAD), insensible a
    # los propios atipicos que se quieren detectar.
    s_i = s.interpolate("time")
    nivel = s_i.rolling(freq_periodos, center=True).mean().rolling(2, center=True).mean().shift(-1)
    razon = (s_i / nivel).dropna()
    irregular = razon / razon.groupby(razon.index.month).transform("median")
    mad = (irregular - irregular.median()).abs().median()
    z = (irregular - irregular.median()) / (1.4826 * mad)
    hallazgos["atipicos"] = {d.strftime("%Y-%m"): round(float(v), 2)
                             for d, v in z[z.abs() > umbral_z].items()}

    print("=" * 74)
    print(f"AUDITORÍA DE INTEGRIDAD — columna '{columna}'")
    print("=" * 74)
    ok = "✔"; no = "✘"
    print(f"{ok if hallazgos['indice_ordenado'] else no} Índice cronológicamente ordenado")
    print(f"{ok if hallazgos['indice_unico'] else no} Índice sin fechas duplicadas")
    print(f"{ok if hallazgos['equiespaciado'] else no} Intervalos equiespaciados "
          f"(frecuencia declarada: {freq_esperada})")
    print(f"{ok if not hallazgos['fechas_ausentes'] else no} Calendario completo "
          f"— fechas ausentes: {hallazgos['fechas_ausentes'] or 'ninguna'}")
    print(f"{ok if hallazgos['n_nulos'] == 0 else '⚠'} Valores nulos: "
          f"{hallazgos['n_nulos']} {hallazgos['fechas_nulas']}")
    print(f"{ok if hallazgos['n_no_positivos'] == 0 else no} Valores no positivos: "
          f"{hallazgos['n_no_positivos']}")
    print(f"⚠ Candidatos a atípico (z robusto del componente irregular, |z| > {umbral_z}):")
    if hallazgos["atipicos"]:
        for fecha, valor in hallazgos["atipicos"].items():
            signo = "por encima" if valor > 0 else "por debajo"
            print(f"      {fecha}  z = {valor:+.2f}  ({signo} de lo esperado para ese mes)")
    else:
        print("      ninguno")
    print("\nNota: un 'candidato a atípico' es una señal para INVESTIGAR, no para eliminar.")
    print("Contraste cada uno con los antecedentes del §3 antes de decidir qué hacer.")
    return hallazgos


auditoria = auditar_serie(df, "consumo_gwh")

### 🧭 Punto de decisión 1 — Valores faltantes

La auditoría identifica tres meses sin registro. **Un valor faltante no es un cero.** Antes de imputar, decida y justifique:

| Opción | Cuándo es razonable | Riesgo |
|---|---|---|
| Interpolación lineal | Huecos aislados en series suaves | Ignora la estacionalidad: mal si el hueco cae en un pico |
| Interpolación temporal (`method="time"`) | Índice con espaciado irregular | Mismo riesgo anterior |
| Imputación estacional (valor del mismo mes del año anterior, ajustado por tendencia) | Series con estacionalidad marcada | Propaga el ruido de ese año |
| Descomposición e imputación del residuo | Series largas y bien estructuradas | Más complejo; requiere serie ya descompuesta |
| Dejar el `NaN` | Modelos de espacio de estados los manejan de forma nativa | No todas las funciones lo toleran |

> **Lo que se evalúa:** que usted mire *en qué mes* cae cada hueco antes de elegir. Interpolar linealmente un enero —mes de consumo máximo— entre un diciembre alto y un febrero alto es defendible; interpolar un julio entre junio y agosto también. Los tres huecos de esta serie no son equivalentes entre sí.

Y, sobre todo: **declare la imputación**. Una serie imputada tiene autocorrelación artificial en los rezagos vecinos al hueco, y eso afecta el diagnóstico posterior.

In [ ]:
# Comparación de estrategias de imputación (NO es la decisión final: es el insumo para tomarla)
s = df["consumo_gwh"]
huecos = s[s.isna()].index

estrategias = {
    "lineal": s.interpolate("linear"),
    "temporal": s.interpolate("time"),
    "estacional (mes anterior año previo, ajustado)":
        s.fillna(s.shift(12) * (s.shift(1) / s.shift(13))),
}

comparacion = pd.DataFrame({k: v.loc[huecos] for k, v in estrategias.items()}).round(2)
comparacion.insert(0, "mes del año", huecos.month)
print("Valor imputado según estrategia, para cada hueco:\n")
display(comparacion)

fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.0))
for ax, hueco in zip(axes, huecos):
    tramo = slice(hueco - pd.DateOffset(months=7), hueco + pd.DateOffset(months=7))
    ax.plot(s.loc[tramo].index, s.loc[tramo].values, "o-", color="#cccccc", ms=4,
            label="observado")
    for (nombre, serie), color in zip(estrategias.items(), ["#b30338", "#0057b8", "#2e7d32"]):
        ax.plot([hueco], [serie.loc[hueco]], "o", ms=8, color=color, label=nombre)
    ax.set_title(f"Hueco: {hueco:%Y-%m}")
    ax.tick_params(axis="x", rotation=45, labelsize=7)
axes[0].legend(fontsize=6.5, loc="best")
plt.tight_layout(); plt.show()

print("Observe que las tres estrategias NO coinciden, y que la diferencia entre ellas")
print("depende del mes en que cae el hueco. Ahí está la decisión que debe fundamentar.")

---
# 6. Primera caracterización visual

A partir de aquí trabajamos con una versión imputada de la serie. **Para este documento base** se adopta la interpolación temporal; en su entrega de la Semana 1 usted debe **elegir y justificar** su propia estrategia, que puede coincidir o no con esta.

In [ ]:
y = df["consumo_gwh"].interpolate("time")
y.name = "consumo_gwh"

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True,
                         gridspec_kw={"height_ratios": [2, 1]})

axes[0].plot(y.index, y.values, color="#b30338", lw=1.3)
axes[0].plot(huecos, y.loc[huecos], "o", color="#0057b8", ms=6,
             label="valores imputados", zorder=5)
axes[0].axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2021-12-01"),
                color="#999999", alpha=0.18, label="periodo COVID-19")
axes[0].axvline(pd.Timestamp("2019-04-01"), color="#2e7d32", ls="--", lw=1.2,
                label="incorporación de clientes")
axes[0].set_ylabel("GWh")
axes[0].set_title("Consumo mensual del segmento comercial — DEL, 2015–2024")
axes[0].legend(fontsize=8, ncol=3)

axes[1].plot(y.index, (y / y.shift(12) - 1) * 100, color="#455a64", lw=1.1)
axes[1].axhline(0, color="k", lw=0.8)
axes[1].set_ylabel("% var. interanual")
axes[1].set_title("Variación respecto del mismo mes del año anterior", loc="left")

plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.4))

# --- Perfil estacional por mes ---
perfil = y.groupby(y.index.month).mean() / y.mean()
axes[0].bar(perfil.index, perfil.values, color="#b30338", alpha=0.8)
axes[0].axhline(1, color="k", lw=0.9)
axes[0].set_xticks(range(1, 13))
axes[0].set_xlabel("mes"); axes[0].set_ylabel("índice (media anual = 1)")
axes[0].set_title("Perfil estacional promedio")

# --- Gráfico estacional superpuesto: un trazo por año ---
for anio, grupo in y.groupby(y.index.year):
    color = "#cccccc" if anio not in (2020, 2024) else ("#0057b8" if anio == 2020 else "#b30338")
    axes[1].plot(grupo.index.month, grupo.values, color=color, lw=1.4 if color != "#cccccc" else 0.9,
                 label=str(anio) if color != "#cccccc" else None)
axes[1].set_xticks(range(1, 13)); axes[1].set_xlabel("mes")
axes[1].set_title("Superposición anual (2020 y 2024 destacados)")
axes[1].legend(fontsize=8)

# --- Estabilidad de media y varianza ---
axes[2].plot(y.rolling(12).mean(), color="#b30338", label="media móvil 12m")
axes[2].plot(y.rolling(12).std(), color="#0057b8", label="desv. estándar móvil 12m")
axes[2].set_title("¿Media y varianza constantes?")
axes[2].legend(fontsize=8); axes[2].tick_params(axis="x", rotation=30, labelsize=7)

plt.tight_layout(); plt.show()

# Diagnóstico aditivo vs. multiplicativo: amplitud estacional año a año.
# Si la amplitud ABSOLUTA crece mientras la RELATIVA se mantiene estable,
# la estacionalidad es multiplicativa.
amplitud = pd.DataFrame({
    "media anual (GWh)": y.groupby(y.index.year).mean(),
    "amplitud pico-valle (GWh)": y.groupby(y.index.year).max() - y.groupby(y.index.year).min(),
})
amplitud["amplitud relativa (%)"] = (100 * amplitud["amplitud pico-valle (GWh)"]
                                     / amplitud["media anual (GWh)"])
amplitud.index.name = "año"
display(amplitud.round(2).style.format("{:.2f}").set_caption(
    "Amplitud estacional por año (2020 y 2021 distorsionados por el shock)"))

normales = amplitud.drop(index=[2020, 2021])
print(f"Excluyendo 2020-2021:")
print(f"   amplitud absoluta: {normales['amplitud pico-valle (GWh)'].iloc[0]:.1f} GWh (2015) "
      f"→ {normales['amplitud pico-valle (GWh)'].iloc[-1]:.1f} GWh (2024)")
print(f"   amplitud relativa: {normales['amplitud relativa (%)'].iloc[0]:.1f}% (2015) "
      f"→ {normales['amplitud relativa (%)'].iloc[-1]:.1f}% (2024)")
print(f"   desv. estándar de la amplitud relativa: {normales['amplitud relativa (%)'].std():.2f} puntos")

### 🧭 Punto de decisión 2 — ¿Estacionalidad aditiva o multiplicativa?

Compare la desviación estándar móvil con la media móvil. Si la dispersión **crece junto con el nivel**, la estacionalidad es multiplicativa y el logaritmo (o una transformación Box-Cox) es la respuesta natural. Si la dispersión es estable en términos absolutos, es aditiva y no hace falta transformar.

La tabla de amplitudes que acaba de imprimir es la forma cuantitativa de zanjar la discusión. Fíjese en las dos últimas columnas: si la amplitud **absoluta** crece con el nivel mientras la amplitud **relativa** se mantiene estable, la estructura es multiplicativa y el logaritmo la convertirá en aditiva. Si ocurriera lo contrario —amplitud absoluta estable y relativa decreciente— sería aditiva y transformar sería un error.

Esta decisión pertenece formalmente a la **Semana 2**, pero conviene que la anticipe ahora.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.4))
axes[0].plot(y.index, y.values, color="#b30338", lw=1.1)
axes[0].set_title("Serie (nivel)")
plot_acf(y.values, lags=48, ax=axes[1], title="ACF — 48 rezagos")
plot_pacf(y.values, lags=48, ax=axes[2], method="ywm", title="PACF — 48 rezagos")
plt.tight_layout(); plt.show()

print("ACF en los rezagos estacionales:")
r = acf(y.values, nlags=48)
for k in (1, 6, 12, 24, 36, 48):
    print(f"   rezago {k:>2}: ρ = {r[k]:+.3f}")
print("\nDecaimiento lento en todos los rezagos + picos marcados en 12, 24, 36 y 48:")
print("la serie NO es estacionaria y tiene una componente estacional anual clara.")
print("Este correlograma NO sirve para identificar órdenes: eso viene tras estacionarizar.")

### Relación con las covariables

Antes de descartar el enfoque multivariado, vale la pena mirar qué explican las covariables. Preste atención a un matiz: **una correlación marginal baja no significa que la variable sea irrelevante**, porque puede estar dominada por la estacionalidad.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.2))
for ax, var, etiqueta in zip(
        axes,
        ["temperatura_media_c", "clientes_comerciales", "dias_habiles"],
        ["Temperatura media (°C)", "Clientes comerciales", "Días hábiles"]):
    ax.scatter(df[var], y, s=16, alpha=0.55, color="#b30338")
    rho = np.corrcoef(df[var].values, y.values)[0, 1]
    ax.set_xlabel(etiqueta); ax.set_ylabel("consumo (GWh)")
    ax.set_title(f"ρ = {rho:+.3f}")
plt.tight_layout(); plt.show()

print("Lecturas:")
print(" · Temperatura: relación fuerte pero NO monótona — el consumo sube tanto con")
print("   calor (refrigeración) como con frío (calefacción). Un ρ lineal la subestima.")
print(" · Clientes: correlacionado con el consumo, pero ambos crecen en el tiempo.")
print("   Correlación entre dos series con tendencia ≠ relación causal (regresión espuria).")
print(" · Días hábiles: ρ casi nulo. El efecto existe, pero queda enmascarado por la")
print("   estacionalidad, mucho más grande. Se detecta sobre la serie ya desestacionalizada.")

---
# 7. Partición cronológica y sellado del conjunto de prueba

> ### 🚫 Regla inviolable del curso
> **Nunca use `train_test_split` con mezcla aleatoria en una serie de tiempo.** Entrenar con datos posteriores a los que se predicen es fuga de información (*data leakage*) y produce métricas ficticias que se derrumban en producción.

La partición de este caso queda fijada aquí y es la misma para todos:

| Conjunto | Periodo | Observaciones | Uso |
|---|---|---|---|
| **Entrenamiento** | 2015-01 a 2023-12 | 108 | Exploración, transformaciones, identificación y estimación |
| **Prueba** | 2024-01 a 2024-12 | 12 | **Solo** evaluación final, en la Semana 4 |

> ### 🔒 Sellado del conjunto de prueba
> Durante las semanas 1 a 3, el año 2024 **no debe mirarse**: ni graficarse, ni usarse para elegir órdenes, ni consultarse "solo para ver". Ese año existe para responder una pregunta que solo tiene sentido una vez: *¿qué tan bien pronostica este modelo datos que nunca vio?*
>
> Mirarlo antes no es un error técnico menor; **invalida la métrica final**.
>
> El horizonte de prueba, $h = 12$, no es arbitrario: coincide con el horizonte de la decisión de negocio descrita en §3.

In [ ]:
CORTE = "2023-12-31"
H = 12                       # horizonte de pronóstico, en meses

y_train = y.loc[:CORTE]
y_test = y.loc["2024-01-01":]
X_train = df.loc[:CORTE, ["clientes_comerciales", "temperatura_media_c", "dias_habiles"]]
X_test = df.loc["2024-01-01":, ["clientes_comerciales", "temperatura_media_c", "dias_habiles"]]

assert len(y_train) == 108 and len(y_test) == H
assert y_train.index.max() < y_test.index.min(), "La partición debe ser cronológica"

print(f"Entrenamiento: {y_train.index.min():%Y-%m} a {y_train.index.max():%Y-%m}  "
      f"({len(y_train)} obs, {len(y_train)/12:.0f} años)")
print(f"Prueba (sellada): {y_test.index.min():%Y-%m} a {y_test.index.max():%Y-%m}  "
      f"({len(y_test)} obs)")

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(y_train.index, y_train.values, color="#b30338", lw=1.2, label="entrenamiento (2015–2023)")
ax.axvspan(y_test.index.min(), y_test.index.max(), color="#455a64", alpha=0.18)
ax.text(pd.Timestamp("2024-06-15"), y_train.max() * 0.98, "2024\nSELLADO",
        ha="center", va="top", fontsize=9, color="#455a64", fontweight="bold")
ax.set_ylabel("GWh"); ax.legend(fontsize=8)
ax.set_title("Partición cronológica del caso transversal")
plt.tight_layout(); plt.show()

---
# 8. Modelos de referencia y métricas

Un modelo solo es bueno **en comparación con algo**. Antes de ajustar cualquier SARIMA, hay que saber cuánto error comete un procedimiento trivial: esa es la vara mínima.

### 8.1 Modelos de referencia obligatorios

| Referencia | Predicción para el mes $t+h$ | Qué mide |
|---|---|---|
| **Media histórica** | $\bar{y}$ del entrenamiento | El aporte de reconocer cualquier estructura |
| **Ingenuo** (*naïve*) | $y_T$, el último valor observado | El aporte de reconocer la estacionalidad |
| **Ingenuo estacional** | $y_{T+h-12}$, el mismo mes del año anterior | **La vara real.** Muy difícil de superar en series estacionales |
| **Deriva** (*drift*) | $y_T + h \cdot \frac{y_T - y_1}{T-1}$ | El aporte de modelar la tendencia por sobre extrapolarla |

> Si su SARIMA no supera al ingenuo estacional, el modelo **no aporta valor** y así debe reportarlo. Esa conclusión, honestamente argumentada, vale más que un resultado inflado.

### 8.2 Métricas

$$\text{RMSE} = \sqrt{\frac{1}{h}\sum_{i=1}^{h}(y_i-\hat{y}_i)^2}
\qquad
\text{MAE} = \frac{1}{h}\sum_{i=1}^{h}|y_i-\hat{y}_i|
\qquad
\text{MAPE} = \frac{100}{h}\sum_{i=1}^{h}\left|\frac{y_i-\hat{y}_i}{y_i}\right|$$

$$\text{MASE} = \frac{\text{MAE del modelo}}{\text{MAE del ingenuo estacional en el conjunto de entrenamiento}}$$

| Métrica | Ventaja | Limitación |
|---|---|---|
| RMSE | Penaliza fuertemente los errores grandes; adecuado cuando un error grande es desproporcionadamente costoso | Sensible a atípicos; en unidades de la serie |
| MAE | Interpretación directa en GWh | Trata todos los errores por igual |
| MAPE | Adimensional, comparable entre series | Explota si hay valores cercanos a cero; penaliza asimétricamente la sobreestimación |
| MASE | Adimensional y con lectura inmediata: **< 1 significa mejor que el ingenuo estacional** | Requiere fijar la referencia |

**Métrica principal del caso:** RMSE, porque el costo de las desviaciones de contrato crece más que proporcionalmente. **Métrica de reporte:** MASE, porque comunica de inmediato si el modelo aporta.

In [ ]:
def metricas(y_real, y_pred, y_entrenamiento, s=12):
    '''RMSE, MAE, MAPE y MASE. MASE usa el naive estacional en entrenamiento como escala.'''
    y_real = np.asarray(y_real, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    e = y_real - y_pred
    yt = np.asarray(y_entrenamiento, dtype=float)
    escala = np.mean(np.abs(yt[s:] - yt[:-s]))
    return {"RMSE": np.sqrt(np.mean(e**2)),
            "MAE": np.mean(np.abs(e)),
            "MAPE (%)": np.mean(np.abs(e / y_real)) * 100,
            "MASE": np.mean(np.abs(e)) / escala}


def referencias(y_train, h=12, s=12):
    '''Genera las predicciones de los cuatro modelos de referencia obligatorios.'''
    v = y_train.to_numpy(dtype=float)
    pendiente = (v[-1] - v[0]) / (len(v) - 1)
    return {
        "Media histórica": np.repeat(v.mean(), h),
        "Ingenuo": np.repeat(v[-1], h),
        "Ingenuo estacional": v[-s:][:h],
        "Deriva": v[-1] + pendiente * np.arange(1, h + 1),
    }


preds_ref = referencias(y_train, h=H)
tabla_ref = pd.DataFrame({nombre: metricas(y_test, p, y_train)
                          for nombre, p in preds_ref.items()}).T
display(tabla_ref.style.format("{:.3f}").set_caption(
    "Desempeño de los modelos de referencia sobre 2024 (h = 12)"))

fig, ax = plt.subplots(figsize=(11, 3.4))
ax.plot(y_train.index[-36:], y_train.values[-36:], color="#333333", lw=1.2, label="entrenamiento")
ax.plot(y_test.index, y_test.values, color="#000000", lw=2.2, label="real 2024")
for (nombre, p), color in zip(preds_ref.items(), ["#9e9e9e", "#ef6c00", "#0057b8", "#2e7d32"]):
    ax.plot(y_test.index, p, "--", color=color, lw=1.4, label=nombre)
ax.set_ylabel("GWh"); ax.legend(fontsize=8, ncol=3)
ax.set_title("Modelos de referencia frente a 2024 — esta es la vara a superar")
plt.tight_layout(); plt.show()

mejor = tabla_ref["RMSE"].idxmin()
print(f"Mejor referencia: {mejor} — RMSE = {tabla_ref.loc[mejor, 'RMSE']:.2f} GWh "
      f"({tabla_ref.loc[mejor, 'MAPE (%)']:.2f}% de error medio).")
print("Su modelo de la Semana 4 debe superar esta cifra o explicar convincentemente por qué no.")

> ### 📌 Nota sobre esta sección
> Las métricas anteriores usan el año 2024 **solo para mostrarle cómo se calculan y cuál es la vara**. Es la única vez que el conjunto sellado aparece antes de la Semana 4, y aparece con modelos que no requieren ninguna decisión suya. Nada de lo que verá aquí le permite ajustar órdenes o transformaciones: no hay fuga.
>
> A partir de la próxima celda, el 2024 vuelve a estar sellado.

---
# 9. Las decisiones que deberá fundamentar

Esta es la sección operativa del documento. Cada fila es una decisión que **debe aparecer explícitamente** en la entrega correspondiente, con su evidencia.

## Semana 1 — Caracterización de la serie

| # | Decisión | Evidencia mínima exigida |
|---|---|---|
| 1.1 | Estrategia de imputación de los tres huecos | Comparación de al menos dos estrategias y justificación por el mes en que cae cada hueco |
| 1.2 | Tratamiento del dato atípico de 2017-11 | Gráfico que lo aísle + decisión (conservar / marcar / ajustar) y su efecto sobre la media y la varianza |
| 1.3 | Delimitación del periodo COVID-19 | Fechas de inicio y término del efecto, sustentadas en la variación interanual |
| 1.4 | Componentes presentes: tendencia, estacionalidad, ciclo, quiebres | Gráfico de la serie, perfil estacional, superposición anual y ACF |
| 1.5 | Enfoque univariado o multivariado | Análisis de las tres covariables y justificación de la elección |

## Semana 2 — Estacionariedad y transformaciones

| # | Decisión | Evidencia mínima exigida |
|---|---|---|
| 2.1 | Estacionalidad aditiva o multiplicativa | Media y desviación móviles, coeficiente de variación por tramos |
| 2.2 | Transformación estabilizadora: ninguna, logaritmo o Box-Cox | Comparación antes/después y, si usa Box-Cox, el λ estimado |
| 2.3 | Orden de diferenciación estacional $D$ | ADF y KPSS antes y después; ACF en los rezagos 12, 24, 36 |
| 2.4 | Orden de diferenciación regular $d$ | ADF, KPSS y **criterio de varianza** para descartar sobre-diferenciación |
| 2.5 | Tratamiento del quiebre COVID en la serie transformada | Comparación entre modelar con variable indicadora vs. acotar la muestra |

## Semana 3 — Identificación, estimación y validación

| # | Decisión | Evidencia mínima exigida |
|---|---|---|
| 3.1 | Órdenes candidatos $(p,q)$ y $(P,Q)$ | Lectura argumentada de ACF y PACF de la serie ya estacionaria |
| 3.2 | Modelo seleccionado | Tabla comparativa de al menos 4 especificaciones con AIC y BIC |
| 3.3 | Validación de residuos | ACF de residuos, Ljung-Box, normalidad y homocedasticidad |
| 3.4 | Qué hacer si los residuos fallan | Diagnóstico del patrón remanente y respecificación documentada |
| 3.5 | Parsimonia vs. ajuste | Justificación si elige un modelo con AIC ligeramente peor pero más simple |

## Semana 4 — Pronóstico, evaluación y recomendación

| # | Decisión | Evidencia mínima exigida |
|---|---|---|
| 4.1 | Retransformación del pronóstico a la escala original | Tratamiento explícito del sesgo de retransformación del logaritmo |
| 4.2 | Nivel de confianza de los intervalos | Justificación en función del costo asimétrico descrito en §3 |
| 4.3 | Comparación con los cuatro modelos de referencia | Tabla con RMSE, MAE, MAPE y MASE para todos |
| 4.4 | Veredicto sobre el aporte del modelo | MASE y lectura explícita: ¿aporta o no aporta? |
| 4.5 | **Recomendación de volumen a contratar** | Cifra mes a mes, con el criterio de decisión que usó frente a la asimetría del costo |
| 4.6 | Limitaciones y riesgos del pronóstico | Al menos tres, con su impacto potencial sobre la decisión |

> ### ⚖️ Sobre la decisión 4.5
> Es la única que no es estadística. Si sobreestimar cuesta 1 y subestimar cuesta 3, el volumen óptimo **no es el pronóstico puntual**: es un cuantil superior de la distribución predictiva. Que usted llegue o no a esa conclusión es lo que distingue un ejercicio académico de una recomendación profesional.

---
# 10. Reglas de entrega y reproducibilidad

### Formato

- Un notebook por semana: `MCDI505_S{n}_ApellidoNombre.ipynb`, más su exportación a HTML o PDF.
- El notebook debe ejecutarse completo y sin errores desde un kernel reiniciado (*Kernel → Restart & Run All*). Un notebook que no corre se evalúa como no logrado en el criterio de reproducibilidad.
- Rutas **relativas**, nunca absolutas. `pd.read_csv("datos/consumo_electrico_litoral.csv")`, no `C:/Users/...`.
- Semilla fija declarada al inicio en todo procedimiento aleatorio.
- No modifique el archivo de datos original. Cualquier limpieza se hace en código, dentro del notebook, de modo que sea auditable.

### Estructura sugerida de cada entrega

```
1. Objetivo de la semana         (2-3 líneas)
2. Decisiones tomadas            (tabla: decisión / evidencia / alternativa descartada)
3. Desarrollo                    (código + gráficos + interpretación intercalada)
4. Conclusiones                  (qué quedó establecido y qué hereda la semana siguiente)
5. Limitaciones y pendientes
```

### Sobre el uso de asistentes de IA

Está permitido y es esperable que utilice asistentes de IA como apoyo. Se exige, eso sí, que:

1. **Declare** en qué partes los usó, en una nota al final del notebook.
2. **Verifique** todo resultado que le entreguen. Los asistentes producen con soltura código que corre y conclusiones que no se sostienen; en series de tiempo, confundir la diferencia estacional con `np.diff(x, n=12)` es un error frecuente y silencioso.
3. **Sea capaz de justificar por escrito** cualquier decisión del notebook. La fundamentación es lo que se evalúa, y no se delega.

---


---
# 11. Entregable de la Semana 1

**Modalidad:** individual · **Archivo:** `MCDI505_S1_ApellidoNombre.ipynb` + HTML  
**Carácter:** formativo con retroalimentación docente dentro de 5 días hábiles  

Complete las celdas marcadas con `# TODO`. Recuerde: cada respuesta escrita debe seguir el formato **decisión / evidencia / alternativa descartada / consecuencia**.

In [ ]:
# =============================================================================
# EJERCICIO 1 — Auditoría e imputación  (criterios 1 y 3)
# =============================================================================

# TODO 1.1 — Ejecute auditar_serie() sobre la serie y comente cada hallazgo.


# TODO 1.2 — Para cada uno de los tres huecos, indique en qué mes del año cae y
#            qué implica eso. Elija UNA estrategia de imputación y justifíquela.
#            Sugerencia: no tiene por qué usar la misma estrategia en los tres.


# TODO 1.3 — Cuantifique el impacto de su decisión: ¿cuánto cambian la media,
#            la desviación estándar y la ACF en el rezago 1 según la estrategia?


# --- Respuesta escrita ---
# Decisión:
# Evidencia:
# Alternativa descartada:
# Consecuencia para las semanas siguientes:

In [ ]:
# =============================================================================
# EJERCICIO 2 — El atípico de noviembre de 2017  (criterio 3)
# =============================================================================

# TODO 2.1 — Aísle el dato: grafique 2017 completo junto a 2016 y 2018 superpuestos.
#            ¿Cuán anómalo es en términos de su variación interanual?


# TODO 2.2 — Cuantifique: ¿en cuántas desviaciones estándar de la variación
#            interanual se aparta del resto de la serie?


# TODO 2.3 — Decida: ¿conservar, marcar con una variable indicadora, o reemplazar?
#            El área de operaciones confirmó que se trata de una recalibración de
#            medidores, no de un consumo real. ¿Cambia eso su decisión?


# --- Respuesta escrita ---
# Decisión:
# Evidencia:
# Alternativa descartada:
# Consecuencia:

In [ ]:
# =============================================================================
# EJERCICIO 3 — Delimitación del efecto COVID-19  (criterios 3 y 13)
# =============================================================================

# TODO 3.1 — Usando la variación interanual, determine el mes de inicio del
#            efecto, el mes de mayor caída y el mes en que el consumo vuelve a
#            su trayectoria previa. Justifique el criterio que usó para "volver".


# TODO 3.2 — Estime la magnitud: ¿cuánta energía dejó de consumirse respecto de
#            lo que cabía esperar? Proponga y aplique un contrafactual simple.


# TODO 3.3 — Anticipe el problema de la Semana 3: la diferenciación estacional
#            compara cada mes con el mismo mes del año anterior. ¿Qué ocurre
#            cuando esa comparación cruza el borde del shock? ¿Qué observaciones
#            de la serie diferenciada quedarán distorsionadas?


# --- Respuesta escrita ---
# Ventana del efecto:
# Magnitud estimada:
# Implicancia para el modelamiento:

In [ ]:
# =============================================================================
# EJERCICIO 4 — Caracterización y enfoque  (criterios 2 y 13)
# =============================================================================

# TODO 4.1 — Enumere los componentes presentes en la serie usando terminología
#            precisa. Para cada uno, señale el gráfico que lo evidencia.
#            Atención: ¿hay ciclo, o solo estacionalidad? No son lo mismo.


# TODO 4.2 — Sobre el perfil estacional: el consumo tiene DOS máximos anuales.
#            Explique el mecanismo físico detrás de cada uno y verifíquelo
#            contra la variable temperatura_media_c.


# TODO 4.3 — Decida el enfoque (univariado o multivariado) y justifique.
#            Si descarta las covariables, explique por qué; si las conserva,
#            indique cuál usaría y qué problema anticipa al necesitar sus
#            valores FUTUROS para pronosticar.


# TODO 4.4 — Redacte tres limitaciones del análisis que ya son visibles.


# --- Respuesta escrita ---
# Componentes identificados:
# Mecanismo de la doble estacionalidad:
# Enfoque elegido y justificación:
# Limitaciones:

---
# 12. Uso excepcional de un conjunto de datos propio

El caso transversal está diseñado sobre la serie provista, y esa es la vía estándar. Sin embargo, si usted trabaja con series temporales en su ámbito profesional y desea aplicar el proyecto a datos de su organización, puede solicitarlo.

### Requisitos técnicos del conjunto alternativo

| Requisito | Mínimo exigido |
|---|---|
| Longitud | 100 observaciones; **al menos 4 ciclos estacionales completos** si tiene estacionalidad |
| Frecuencia | Regular y equiespaciada, sin cambios de frecuencia a lo largo del periodo |
| Faltantes | Menos del 10% del total y sin bloques contiguos extensos |
| Estructura | Debe presentar tendencia y/o estacionalidad: una serie que ya es ruido blanco no permite desarrollar las cuatro semanas |
| Horizonte | Debe admitir un conjunto de prueba final de al menos 10% de las observaciones |

> El requisito de **4 ciclos estacionales** es más restrictivo que el de 100 observaciones y suele ser el que descarta candidatos. Con datos mensuales significan 48 observaciones como piso absoluto; con datos trimestrales, 16, lo que rara vez basta.

### Protocolo de aprobación

1. Solicitar por el foro de la asignatura **antes del cierre de la Semana 1**, adjuntando: fuente, periodo, frecuencia, longitud, gráfico de la serie y la pregunta de decisión que busca responder.
2. Esperar aprobación escrita del equipo docente. Una solicitud enviada no es una solicitud aprobada.
3. Si los datos son **confidenciales**, acordar previamente uno de estos mecanismos: anonimización de identificadores, reescalamiento de la variable objetivo por un factor no revelado, o desplazamiento temporal del índice. Ninguno de los tres altera la estructura de dependencia ni, por lo tanto, la validez del análisis.

### Fuentes públicas sugeridas, si opta por esta vía

| Fuente | Fortaleza |
|---|---|
| Banco Central de Chile — Base de Datos Estadísticos | Series macroeconómicas chilenas, largas y bien documentadas |
| Instituto Nacional de Estadísticas (INE) | IPC, empleo, producción sectorial |
| Coordinador Eléctrico Nacional | Generación y demanda del sistema eléctrico chileno |
| FRED (Reserva Federal de St. Louis) | Amplitud y calidad de metadatos |
| Kaggle | Variedad temática; **verifique siempre la procedencia** antes de usar |

> ⚠️ Advertencia sobre datos financieros diarios. Los precios de acciones y criptomonedas son atractivos pero mal candidatos para este proyecto: son aproximadamente caminatas aleatorias, de modo que el mejor pronóstico posible es el último valor observado y ningún ARIMA lo supera. Si le interesa esa familia de series, la variable a modelar es la **volatilidad de los retornos**, lo que corresponde a modelos GARCH y excede el alcance de estas cuatro semanas.

---
# 13. Preguntas frecuentes

**¿Puedo usar `auto_arima` o alguna búsqueda automática de órdenes?**  
Sí, como **contraste**, nunca como sustituto. La rúbrica evalúa que usted lea la ACF y la PACF y proponga órdenes con argumentos. Si `auto_arima` coincide con su propuesta, dígalo; si difiere, la discrepancia es material valioso para su análisis.

**¿Qué hago si mi modelo no supera al ingenuo estacional?**  
Reportarlo. Es un resultado legítimo y frecuente en series con estacionalidad muy regular. Lo que se evalúa es que lo detecte, lo diagnostique y explique qué habría hecho falta.

**¿Debo eliminar el periodo COVID del análisis?**  
Es una decisión suya, y es la más interesante del caso. Ambas rutas —modelar el shock con variables indicadoras o acotar la muestra— son defendibles, y ambas tienen costos: la primera consume grados de libertad y exige suponer una forma funcional para el shock; la segunda reduce la serie a menos de tres ciclos estacionales completos. Elija y argumente.

**¿Cuánto detalle esperan en las respuestas escritas?**  
Entre cinco y quince líneas por decisión. Se evalúa la calidad del argumento, no su extensión.

**¿Puedo trabajar en grupo?**  
Las entregas son individuales. El foro de la asignatura está abierto para discutir criterios y enfoques, lo que se recomienda expresamente. Lo que no puede compartirse es el notebook.

---
# 14. Glosario del caso

| Término | Significado en este caso |
|---|---|
| **Segmento comercial** | Clientes no residenciales ni industriales: comercio, oficinas, hotelería, servicios |
| **GWh** | Gigavatio-hora, unidad de energía. 1 GWh = 1.000 MWh |
| **Mercado spot** | Mercado de compraventa de energía en tiempo real, de precio volátil |
| **Empalme** | Punto de conexión física de un cliente a la red de distribución |
| **Horizonte de pronóstico ($h$)** | Cantidad de periodos hacia adelante que se predicen. Aquí, $h = 12$ |
| **Conjunto sellado** | Porción final de la serie que no se observa hasta la evaluación |
| **Contrafactual** | Trayectoria estimada que la serie habría seguido de no ocurrir un evento |
| **Sesgo de retransformación** | Sesgo que aparece al aplicar $\exp(\cdot)$ a un pronóstico hecho en logaritmos |

---

## Referencias

Box, G. E. P., Jenkins, G. M., Reinsel, G. C., & Ljung, G. M. (2015). *Time series analysis: Forecasting and control* (5ª ed.). Wiley.

Hyndman, R. J., & Athanasopoulos, G. (2021). *Forecasting: Principles and practice* (3ª ed.). OTexts. https://otexts.com/fpp3/

Hyndman, R. J., & Koehler, A. B. (2006). Another look at measures of forecast accuracy. *International Journal of Forecasting, 22*(4), 679–688.

Kulkarni, A. R., Shivananda, A., Sharma, N. R., & Kulkarni, A. (2023). *Time series algorithms recipes: Implement machine learning and deep learning techniques with Python*. Apress.

Tashman, L. J. (2000). Out-of-sample tests of forecasting accuracy: An analysis and review. *International Journal of Forecasting, 16*(4), 437–450.

Woodward, W. A., Sadler, B. P., & Robertson, S. (2022). *Time series for data science: Analysis and forecasting*. Chapman & Hall/CRC.

---

<div style="text-align:center; color:#777; font-size:0.85em;">
MCDI505 — Análisis de Series de Tiempo · Caso Transversal · Magíster en Ciencia de Datos · UNAB Online
</div>